[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/Time-Series-Analysis/blob/main/EN/Seminar_Notebooks/chapter13_seminar_notebook.ipynb)

---

# Chapter 13 Seminar: LPPL Models for Bubble Detection

**Course:** Time Series Analysis and Forecasting  
**Program:** Bachelor program, Faculty of Cybernetics, Statistics and Economic Informatics, Bucharest University of Economic Studies, Romania  
**Academic Year:** 2025-2026

---

## Seminar Objectives

In this practical seminar, you will:
1. Detect super-exponential growth in asset prices
2. Understand and implement the Log-Periodic Power Law (LPPL) model
3. Fit LPPL to real bubble data using partial linearization and differential evolution
4. Validate fits using the 8 filter conditions from the literature
5. Construct bootstrap confidence intervals for the critical time $t_c$
6. Build a rolling LPPLS Confidence Indicator for real-time bubble monitoring
7. Apply LPPL as a negative control on non-bubble crash data
8. Design a risk management strategy based on LPPL signals

## Setup

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !pip install yfinance scipy statsmodels arch -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
from scipy.optimize import differential_evolution, minimize
from scipy import stats
import statsmodels.api as sm
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

COLORS = {
    'blue': '#1A3A6E', 'red': '#DC3545', 'green': '#2E7D32',
    'amber': '#B5853F', 'orange': '#E6802E', 'purple': '#8E44AD',
    'gray': '#666666', 'light_blue': '#5B8BD4'
}

plt.rcParams.update({
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.facecolor': 'none',
    'figure.facecolor': 'none',
    'savefig.facecolor': 'none',
    'legend.frameon': False
})

print('Setup complete. Libraries loaded.')

---
# Part I: Fundamentals (Year 3 Level)

**Objectives:** Detect super-exponential growth, understand the LPPL equation and its components, and fit the model to real bubble data.

**Dataset:** Bitcoin (BTC-USD) from September 2020 to November 2021 — the run-up to the all-time high near $69,000.

## Exercise 1: Super-Exponential Growth Detection

**Task:** Download BTC-USD from Sep 2020 to Nov 2021 using yfinance.  
Plot price, log-price, daily returns, and rolling volatility.  
Then test for super-exponential growth by fitting linear, quadratic, and exponential models to log-price.  
If $R^2$ is substantially higher for the nonlinear models, this is evidence that growth is faster-than-exponential.

In [ ]:
# Download BTC-USD data
btc = yf.download('BTC-USD', start='2020-09-01', end='2021-11-15', progress=False)
btc = btc[['Close']].dropna()
btc.columns = ['Price']
btc['LogPrice'] = np.log(btc['Price'])
btc['Return'] = btc['LogPrice'].diff()
btc['RollingVol'] = btc['Return'].rolling(30).std() * np.sqrt(252)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Panel 1: Price
axes[0, 0].plot(btc.index, btc['Price'], color=COLORS['blue'], linewidth=1)
axes[0, 0].set_title('BTC-USD Price', fontweight='bold')
axes[0, 0].set_ylabel('Price (USD)')
axes[0, 0].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
axes[0, 0].xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))

# Panel 2: Log-price
axes[0, 1].plot(btc.index, btc['LogPrice'], color=COLORS['green'], linewidth=1)
axes[0, 1].set_title('Log-Price $\\ln P(t)$', fontweight='bold')
axes[0, 1].set_ylabel('Log-price')
axes[0, 1].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
axes[0, 1].xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))

# Panel 3: Daily returns
axes[1, 0].plot(btc.index, btc['Return'], color=COLORS['red'], linewidth=0.5, alpha=0.7)
axes[1, 0].axhline(0, color=COLORS['gray'], linestyle='--', linewidth=0.8)
axes[1, 0].set_title('Daily Log-Returns', fontweight='bold')
axes[1, 0].set_ylabel('Log-return')
axes[1, 0].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
axes[1, 0].xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))

# Panel 4: Rolling volatility
axes[1, 1].plot(btc.index, btc['RollingVol'], color=COLORS['purple'], linewidth=1)
axes[1, 1].set_title('30-Day Rolling Volatility (Annualized)', fontweight='bold')
axes[1, 1].set_ylabel('Volatility')
axes[1, 1].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
axes[1, 1].xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))

for ax in axes.flatten():
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_facecolor('none')
fig.patch.set_alpha(0)
plt.tight_layout()
plt.show()

print(f'Period: {btc.index[0].strftime("%Y-%m-%d")} to {btc.index[-1].strftime("%Y-%m-%d")}')
print(f'Observations: {len(btc):,}')
print(f'Price range: ${btc["Price"].min():,.0f} to ${btc["Price"].max():,.0f}')
print(f'Total return: {(btc["Price"].iloc[-1]/btc["Price"].iloc[0] - 1)*100:.1f}%')

In [ ]:
# Test for super-exponential growth: fit linear, quadratic, exponential to log-price
y = btc['LogPrice'].values
t = np.arange(len(y))

# Model 1: Linear (exponential price growth) -> ln P = a + b*t
X_lin = sm.add_constant(t)
mod_lin = sm.OLS(y, X_lin).fit()
y_lin = mod_lin.predict(X_lin)

# Model 2: Quadratic (super-exponential) -> ln P = a + b*t + c*t^2
X_quad = sm.add_constant(np.column_stack([t, t**2]))
mod_quad = sm.OLS(y, X_quad).fit()
y_quad = mod_quad.predict(X_quad)

# Model 3: Exponential fit to log-price -> ln P = a + b*exp(c*t)
# We use nonlinear least squares via scipy
from scipy.optimize import curve_fit

def exp_model(t, a, b, c):
    return a + b * np.exp(c * t)

try:
    popt_exp, _ = curve_fit(exp_model, t, y, p0=[y[0], 0.01, 0.005], maxfev=10000)
    y_exp = exp_model(t, *popt_exp)
    ss_res_exp = np.sum((y - y_exp)**2)
    ss_tot = np.sum((y - y.mean())**2)
    r2_exp = 1 - ss_res_exp / ss_tot
except Exception:
    y_exp = y_quad  # fallback
    r2_exp = 0.0

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Fits
axes[0].plot(t, y, color=COLORS['gray'], linewidth=1, alpha=0.6, label='Observed log-price')
axes[0].plot(t, y_lin, color=COLORS['blue'], linewidth=2, linestyle='--',
             label=f'Linear ($R^2$={mod_lin.rsquared:.4f})')
axes[0].plot(t, y_quad, color=COLORS['red'], linewidth=2, linestyle='-.',
             label=f'Quadratic ($R^2$={mod_quad.rsquared:.4f})')
axes[0].plot(t, y_exp, color=COLORS['green'], linewidth=2, linestyle=':',
             label=f'Exponential ($R^2$={r2_exp:.4f})')
axes[0].set_title('Super-Exponential Growth Test', fontweight='bold')
axes[0].set_xlabel('Trading day (t)')
axes[0].set_ylabel('Log-price')
axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2, frameon=False)
axes[0].spines[['top', 'right']].set_visible(False)
axes[0].set_facecolor('none')

# Panel 2: R-squared comparison
models = ['Linear', 'Quadratic', 'Exponential']
r2_values = [mod_lin.rsquared, mod_quad.rsquared, r2_exp]
bar_colors = [COLORS['blue'], COLORS['red'], COLORS['green']]
bars = axes[1].bar(models, r2_values, color=bar_colors, alpha=0.8, width=0.5)
for bar, val in zip(bars, r2_values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                 f'{val:.4f}', ha='center', fontweight='bold', fontsize=11)
axes[1].set_title('$R^2$ Comparison: Evidence of Super-Exponential Growth', fontweight='bold')
axes[1].set_ylabel('$R^2$')
axes[1].set_ylim(0.9, 1.001)
axes[1].spines[['top', 'right']].set_visible(False)
axes[1].set_facecolor('none')

fig.patch.set_alpha(0)
plt.tight_layout()
plt.show()

print('Interpretation:')
print(f'  Linear R^2:      {mod_lin.rsquared:.4f} (pure exponential price growth)')
print(f'  Quadratic R^2:   {mod_quad.rsquared:.4f} (super-exponential component)')
print(f'  Exponential R^2: {r2_exp:.4f} (stronger super-exponential)')
print(f'\nThe quadratic/exponential dominance over linear confirms super-exponential growth.')
print('This is the hallmark of a bubble regime amenable to LPPL modelling.')

### Question 1.1

Why does super-exponential growth in log-price imply a bubble?  
What economic mechanism (positive feedback loops, herding) drives faster-than-exponential appreciation?

**YOUR ANSWER HERE**

## Exercise 2: The LPPL Equation

**Task:** Explore the LPPL equation and visualize each of its three components separately using synthetic data with known parameters.

The **Log-Periodic Power Law** model (Sornette, Johansen & Bouchaud, 1996) describes the log-price during a bubble:

$$\ln P(t) = A + B(t_c - t)^m + C(t_c - t)^m \cos\!\left(\omega \ln(t_c - t) - \varphi\right)$$

where:
- $A$ = log-price at the critical time $t_c$ (the expected crash date)
- $B < 0$ = amplitude of the power-law growth (negative because $t_c - t$ shrinks)
- $m \in (0, 1)$ = power-law exponent (super-exponential growth requires $m < 1$)
- $C$ = amplitude of the log-periodic oscillations
- $\omega$ = angular log-frequency of the oscillations
- $\varphi$ = phase shift
- $t_c$ = critical time (most probable time for the regime change)

In [ ]:
# Synthetic LPPL with known parameters
tc_syn = 300
A_syn = 11.0
B_syn = -0.5
m_syn = 0.5
C_syn = 0.04
omega_syn = 8.0
phi_syn = 1.0

t_syn = np.arange(0, tc_syn - 5)
dt = tc_syn - t_syn  # time-to-critical

# Component 1: Power-law trend
power_law = A_syn + B_syn * dt**m_syn

# Component 2: Log-periodic oscillations (modulated by power law)
oscillation = C_syn * dt**m_syn * np.cos(omega_syn * np.log(dt) - phi_syn)

# Full LPPL
lppl_full = power_law + oscillation

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: Power-law trend
axes[0].plot(t_syn, power_law, color=COLORS['blue'], linewidth=2)
axes[0].axvline(tc_syn, color=COLORS['red'], linestyle='--', linewidth=1.5, label=f'$t_c = {tc_syn}$')
axes[0].set_title('Component 1: Power-Law Trend\n$A + B(t_c - t)^m$', fontweight='bold')
axes[0].set_xlabel('Time $t$')
axes[0].set_ylabel('Log-price')
axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=1, frameon=False)

# Panel 2: Log-periodic oscillations only
axes[1].plot(t_syn, oscillation, color=COLORS['purple'], linewidth=1.5)
axes[1].axhline(0, color=COLORS['gray'], linestyle='--', linewidth=0.8)
axes[1].axvline(tc_syn, color=COLORS['red'], linestyle='--', linewidth=1.5, label=f'$t_c = {tc_syn}$')
axes[1].set_title('Component 2: Log-Periodic Oscillations\n$C(t_c-t)^m\\cos(\\omega\\ln(t_c-t)-\\varphi)$',
                   fontweight='bold')
axes[1].set_xlabel('Time $t$')
axes[1].set_ylabel('Oscillation amplitude')
axes[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=1, frameon=False)

# Panel 3: Full LPPL
axes[2].plot(t_syn, lppl_full, color=COLORS['red'], linewidth=1.5, label='Full LPPL')
axes[2].plot(t_syn, power_law, color=COLORS['blue'], linewidth=1, linestyle='--',
             alpha=0.6, label='Power-law envelope')
axes[2].axvline(tc_syn, color=COLORS['red'], linestyle='--', linewidth=1.5, alpha=0.5)
axes[2].set_title('Combined LPPL Signal', fontweight='bold')
axes[2].set_xlabel('Time $t$')
axes[2].set_ylabel('Log-price')
axes[2].legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2, frameon=False)

for ax in axes:
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_facecolor('none')
fig.suptitle('Anatomy of the LPPL Equation', fontweight='bold', y=1.01)
fig.patch.set_alpha(0)
plt.tight_layout()
plt.show()

print('Key observations:')
print(f'  - The power-law trend accelerates as t -> tc (super-exponential growth)')
print(f'  - Oscillations become faster and smaller near tc (log-periodic acceleration)')
print(f'  - The frequency of oscillations increases in calendar time but is constant in log(tc-t)')
print(f'\nSynthetic parameters: A={A_syn}, B={B_syn}, m={m_syn}, C={C_syn}, omega={omega_syn}, phi={phi_syn}, tc={tc_syn}')

### Question 2.1

Why must $B < 0$ and $0 < m < 1$ for the LPPL to describe a bubble?  
What happens to the model if $m \geq 1$ or $B > 0$?

**YOUR ANSWER HERE**

## Exercise 3: LPPL Fitting via Partial Linearization

**Task:** Implement the LPPL fitting procedure using partial linearization.  
For fixed nonlinear parameters $(t_c, m, \omega)$, the remaining parameters $(A, B, C_1, C_2)$ can be estimated by ordinary least squares (OLS), where $C_1 = C\cos\varphi$ and $C_2 = C\sin\varphi$.  
Use `scipy.optimize.differential_evolution` to search over the nonlinear parameters.  
Fit the LPPL to the BTC 2021 bubble data.

In [ ]:
# Prepare data for LPPL fitting
log_price = btc['LogPrice'].values
N = len(log_price)
t_data = np.arange(N)

def lppl_func(t, tc, m, omega, A, B, C1, C2):
    """Full LPPL function with C1/C2 parameterization."""
    dt = tc - t
    dt = np.maximum(dt, 1e-10)  # avoid log(0)
    power = dt**m
    log_dt = np.log(dt)
    return A + B * power + C1 * power * np.cos(omega * log_dt) + C2 * power * np.sin(omega * log_dt)


def lppl_cost(nonlinear_params, t, y):
    """Cost function: for fixed (tc, m, omega), solve linear params by OLS."""
    tc, m, omega = nonlinear_params
    dt = tc - t
    if np.any(dt <= 0):
        return 1e12
    power = dt**m
    log_dt = np.log(dt)
    cos_term = power * np.cos(omega * log_dt)
    sin_term = power * np.sin(omega * log_dt)

    # Design matrix for OLS: y = A + B*power + C1*cos_term + C2*sin_term
    X = np.column_stack([np.ones_like(t), power, cos_term, sin_term])
    try:
        beta, residuals, _, _ = np.linalg.lstsq(X, y, rcond=None)
        y_hat = X @ beta
        sse = np.sum((y - y_hat)**2)
    except Exception:
        return 1e12
    return sse


def fit_lppl(t, y, seed=42):
    """Fit LPPL model using differential evolution + OLS."""
    N = len(t)
    bounds = [
        (N - 5, N + 60),  # tc
        (0.1, 0.9),       # m
        (4.0, 25.0)       # omega
    ]
    result = differential_evolution(
        lppl_cost, bounds, args=(t, y),
        seed=seed, maxiter=500, tol=1e-10,
        popsize=30, mutation=(0.5, 1.5), recombination=0.9
    )
    tc_opt, m_opt, omega_opt = result.x

    # Recover linear params
    dt = tc_opt - t
    power = dt**m_opt
    log_dt = np.log(dt)
    cos_term = power * np.cos(omega_opt * log_dt)
    sin_term = power * np.sin(omega_opt * log_dt)
    X = np.column_stack([np.ones_like(t), power, cos_term, sin_term])
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    A_opt, B_opt, C1_opt, C2_opt = beta

    # Compute C and phi
    C_opt = np.sqrt(C1_opt**2 + C2_opt**2)
    phi_opt = np.arctan2(C2_opt, C1_opt)

    params = {
        'tc': tc_opt, 'm': m_opt, 'omega': omega_opt,
        'A': A_opt, 'B': B_opt, 'C': C_opt, 'phi': phi_opt,
        'C1': C1_opt, 'C2': C2_opt, 'sse': result.fun
    }
    return params


# Fit LPPL to BTC data
print('Fitting LPPL to BTC-USD bubble data...')
params = fit_lppl(t_data, log_price, seed=42)
print('Done.\n')

# Generate fitted values
t_fit = np.arange(N)
y_fit = lppl_func(t_fit, params['tc'], params['m'], params['omega'],
                   params['A'], params['B'], params['C1'], params['C2'])

# Print parameters
print('LPPL Fitted Parameters:')
print('=' * 45)
for key in ['A', 'B', 'C', 'm', 'omega', 'phi', 'tc']:
    print(f'  {key:>5s} = {params[key]:>10.4f}')

# Convert tc to calendar date
tc_days = int(np.round(params['tc']))
if tc_days < len(btc):
    tc_date = btc.index[tc_days]
else:
    tc_date = btc.index[-1] + pd.offsets.BDay(tc_days - len(btc) + 1)
print(f'\n  tc as calendar date: {tc_date.strftime("%Y-%m-%d")}')
print(f'  SSE: {params["sse"]:.6f}')

In [ ]:
# Plot: data vs LPPL fit and residuals
residuals = log_price - y_fit

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Fit
axes[0].plot(btc.index, log_price, color=COLORS['blue'], linewidth=1, alpha=0.7,
             label='Observed $\\ln P(t)$')
axes[0].plot(btc.index, y_fit, color=COLORS['red'], linewidth=2,
             label='LPPL fit')
axes[0].axvline(tc_date, color=COLORS['amber'], linestyle='--', linewidth=1.5,
                label=f'$t_c$ = {tc_date.strftime("%Y-%m-%d")}')
axes[0].set_title('LPPL Fit to BTC-USD Log-Price', fontweight='bold')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Log-price')
axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=3, frameon=False)
axes[0].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
axes[0].spines[['top', 'right']].set_visible(False)
axes[0].set_facecolor('none')

# Panel 2: Residuals
axes[1].plot(btc.index, residuals, color=COLORS['purple'], linewidth=0.8)
axes[1].axhline(0, color=COLORS['gray'], linestyle='--', linewidth=0.8)
axes[1].fill_between(btc.index, residuals, 0, alpha=0.3, color=COLORS['purple'])
axes[1].set_title('LPPL Residuals', fontweight='bold')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Residual')
axes[1].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
axes[1].spines[['top', 'right']].set_visible(False)
axes[1].set_facecolor('none')

fig.patch.set_alpha(0)
plt.tight_layout()
plt.show()

# Goodness of fit
ss_tot = np.sum((log_price - log_price.mean())**2)
r2_lppl = 1 - params['sse'] / ss_tot
print(f'LPPL R-squared: {r2_lppl:.6f}')
print(f'Residual std:   {np.std(residuals):.6f}')

---
# Part II: Validation and Diagnostics (Master Level)

**Objectives:** Validate the LPPL fit using the 8 filter conditions, compare against benchmark models, estimate confidence intervals for $t_c$ via bootstrap, and assess stability across multiple fitting windows.

## Exercise 4: The 8 Filter Conditions

**Task:** Check all 8 filter conditions from the LPPL literature (Sornette et al.) on the fitted parameters.  
Display pass/fail for each and explain why each condition is necessary for a valid bubble signal.

The conditions ensure that:
- The fit is physically meaningful (super-exponential, not sub-exponential)
- The critical time $t_c$ is in a plausible future window
- The oscillation frequency $\omega$ is in the empirically observed range
- The log-periodic oscillations are detectable but not dominant

In [ ]:
def check_filter_conditions(params, N):
    """Check the 8 LPPL filter conditions. Returns list of (name, passed, value, bound)."""
    tc, m, omega = params['tc'], params['m'], params['omega']
    B, C = params['B'], params['C']

    conditions = [
        ('1. B < 0 (super-exponential growth)',
         B < 0, f'{B:.4f}', 'B < 0'),
        ('2. 0.1 <= m <= 0.9 (power-law exponent)',
         0.1 <= m <= 0.9, f'{m:.4f}', '0.1 <= m <= 0.9'),
        ('3. 4 <= omega <= 25 (log-frequency)',
         4 <= omega <= 25, f'{omega:.4f}', '4 <= omega <= 25'),
        ('4. tc > N (crash after last observation)',
         tc > N, f'{tc:.1f}', f'tc > {N}'),
        ('5. tc < N + 0.2*N (crash within 20% horizon)',
         tc < N + 0.2 * N, f'{tc:.1f}', f'tc < {N + 0.2*N:.0f}'),
        ('6. |C| > 0 (oscillations present)',
         abs(C) > 0, f'{C:.6f}', '|C| > 0'),
        ('7. m*|B| > 0 (finite-time singularity strength)',
         m * abs(B) > 0, f'{m*abs(B):.4f}', 'm*|B| > 0'),
        ('8. Oscillations per unit: omega/(2*pi) >= 1',
         omega / (2 * np.pi) >= 1, f'{omega/(2*np.pi):.2f}', '>= 1')
    ]
    return conditions


conditions = check_filter_conditions(params, N)

print('LPPL Filter Conditions Check')
print('=' * 75)
n_pass = 0
for name, passed, value, bound in conditions:
    status = 'PASS' if passed else 'FAIL'
    color_mark = '+' if passed else 'X'
    n_pass += int(passed)
    print(f'  [{color_mark}] {status}  {name}')
    print(f'         Value: {value}  |  Required: {bound}')
print('=' * 75)
print(f'Result: {n_pass}/8 conditions passed')

# Visual summary
fig, ax = plt.subplots(figsize=(10, 4))
labels = [f'C{i+1}' for i in range(8)]
passed_vals = [int(c[1]) for c in conditions]
bar_colors = [COLORS['green'] if p else COLORS['red'] for p in passed_vals]
ax.bar(labels, passed_vals, color=bar_colors, alpha=0.8, width=0.6)
ax.set_ylim(0, 1.3)
ax.set_yticks([0, 1])
ax.set_yticklabels(['FAIL', 'PASS'])
ax.set_title(f'LPPL Filter Conditions: {n_pass}/8 Passed', fontweight='bold')
for i, (name, passed, value, bound) in enumerate(conditions):
    ax.text(i, 1.1, value, ha='center', fontsize=8, color=COLORS['gray'])
ax.spines[['top', 'right']].set_visible(False)
ax.set_facecolor('none')
fig.patch.set_alpha(0)
plt.tight_layout()
plt.show()

### Question 4.1

Why is the condition $B < 0$ essential? What would $B > 0$ mean economically?  
Why do we require $t_c$ to be in a narrow future window rather than arbitrarily far?

**YOUR ANSWER HERE**

## Exercise 5: Benchmark Models — Does LPPL Beat Simpler Models?

**Task:** Fit linear, quadratic, exponential, and power-law models to the same BTC data.  
Compare $R^2$, AIC, and BIC across all models including LPPL.  
The LPPL is only justified if it provides a significantly better fit than simpler alternatives.

In [ ]:
# Fit benchmark models to log-price
y_bench = log_price.copy()
t_bench = np.arange(len(y_bench))
n_obs = len(y_bench)

# Model 1: Linear
X1 = sm.add_constant(t_bench)
m1 = sm.OLS(y_bench, X1).fit()

# Model 2: Quadratic
X2 = sm.add_constant(np.column_stack([t_bench, t_bench**2]))
m2 = sm.OLS(y_bench, X2).fit()

# Model 3: Exponential (nonlinear)
from scipy.optimize import curve_fit

def exp_func(t, a, b, c):
    return a + b * np.exp(c * t)

popt3, _ = curve_fit(exp_func, t_bench, y_bench, p0=[y_bench[0], 0.01, 0.005], maxfev=10000)
y_exp_fit = exp_func(t_bench, *popt3)
k3 = 3

# Model 4: Power-law (no oscillations) -> ln P = A + B*(tc-t)^m
def power_law_cost(params_nl, t, y):
    tc, m = params_nl
    dt = tc - t
    if np.any(dt <= 0):
        return 1e12
    X = np.column_stack([np.ones_like(t), dt**m])
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    return np.sum((y - X @ beta)**2)

res_pl = differential_evolution(power_law_cost, [(N-5, N+60), (0.1, 0.9)],
                                args=(t_bench, y_bench), seed=42)
tc_pl, m_pl = res_pl.x
X_pl = np.column_stack([np.ones(n_obs), (tc_pl - t_bench)**m_pl])
beta_pl = np.linalg.lstsq(X_pl, y_bench, rcond=None)[0]
y_pl_fit = X_pl @ beta_pl
k4 = 4  # tc, m, A, B

# Compute AIC/BIC for all models
def compute_aic_bic(y, y_hat, k):
    n = len(y)
    sse = np.sum((y - y_hat)**2)
    sigma2 = sse / n
    log_lik = -n/2 * (np.log(2*np.pi) + np.log(sigma2) + 1)
    aic = -2*log_lik + 2*k
    bic = -2*log_lik + k*np.log(n)
    r2 = 1 - sse / np.sum((y - y.mean())**2)
    return r2, aic, bic

results = {}
results['Linear'] = compute_aic_bic(y_bench, m1.predict(X1), 2)
results['Quadratic'] = compute_aic_bic(y_bench, m2.predict(X2), 3)
results['Exponential'] = compute_aic_bic(y_bench, y_exp_fit, k3)
results['Power Law'] = compute_aic_bic(y_bench, y_pl_fit, k4)
results['LPPL'] = compute_aic_bic(y_bench, y_fit, 7)

print(f'{"Model":>14s} {"R-squared":>10s} {"AIC":>10s} {"BIC":>10s} {"k":>4s}')
print('-' * 52)
for name, (r2, aic, bic) in results.items():
    k = {'Linear': 2, 'Quadratic': 3, 'Exponential': 3, 'Power Law': 4, 'LPPL': 7}[name]
    print(f'{name:>14s} {r2:>10.6f} {aic:>10.1f} {bic:>10.1f} {k:>4d}')

# Bar chart comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
model_names = list(results.keys())
bar_colors = [COLORS['blue'], COLORS['green'], COLORS['amber'], COLORS['purple'], COLORS['red']]

for ax, metric_idx, metric_name in zip(axes, [0, 1, 2], ['$R^2$', 'AIC', 'BIC']):
    vals = [results[m][metric_idx] for m in model_names]
    bars = ax.bar(model_names, vals, color=bar_colors, alpha=0.8, width=0.6)
    ax.set_title(f'{metric_name} Comparison', fontweight='bold')
    ax.set_ylabel(metric_name)
    ax.tick_params(axis='x', rotation=30)
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_facecolor('none')

fig.suptitle('Model Comparison: Does LPPL Beat Simpler Alternatives?', fontweight='bold', y=1.01)
fig.patch.set_alpha(0)
plt.tight_layout()
plt.show()

## Exercise 6: Bootstrap Confidence Intervals

**Task:** Implement residual resampling bootstrap to estimate the uncertainty in the fitted parameters.  
For each of $n_{boot} = 200$ bootstrap samples, resample the LPPL residuals, add them back to the fitted values, and re-estimate the model.  
Report the 95% confidence interval for $t_c$ in both trading days and calendar dates.

In [ ]:
# Bootstrap confidence intervals
n_boot = 200
boot_params = {'tc': [], 'm': [], 'omega': [], 'B': [], 'C': []}

# Compute residuals from the original fit
resid = log_price - y_fit

print(f'Running {n_boot} bootstrap replications...')
for i in range(n_boot):
    np.random.seed(i)
    # Resample residuals
    resid_boot = np.random.choice(resid, size=len(resid), replace=True)
    y_boot = y_fit + resid_boot
    try:
        p_boot = fit_lppl(t_data, y_boot, seed=i)
        # Only keep reasonable estimates
        if 0.1 <= p_boot['m'] <= 0.9 and N - 10 < p_boot['tc'] < N + 80:
            for key in boot_params:
                boot_params[key].append(p_boot[key])
    except Exception:
        pass
    if (i + 1) % 50 == 0:
        print(f'  {i+1}/{n_boot} done ({len(boot_params["tc"])} valid fits)')

print(f'\nTotal valid bootstrap fits: {len(boot_params["tc"])}')

# Convert to arrays
for key in boot_params:
    boot_params[key] = np.array(boot_params[key])

In [ ]:
# Plot bootstrap distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Panel 1: tc distribution
ax = axes[0, 0]
ax.hist(boot_params['tc'], bins=30, color=COLORS['blue'], alpha=0.6, density=True,
        label='Bootstrap $t_c$')
tc_lo, tc_hi = np.percentile(boot_params['tc'], [2.5, 97.5])
ax.axvline(params['tc'], color=COLORS['red'], linewidth=2, linestyle='--',
           label=f'Point estimate: {params["tc"]:.1f}')
ax.axvline(tc_lo, color=COLORS['amber'], linewidth=1.5, linestyle=':',
           label=f'95% CI: [{tc_lo:.1f}, {tc_hi:.1f}]')
ax.axvline(tc_hi, color=COLORS['amber'], linewidth=1.5, linestyle=':')
ax.set_title('Distribution of $t_c$ (Critical Time)', fontweight='bold')
ax.set_xlabel('$t_c$ (trading days)')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=1, frameon=False, fontsize=9)

# Panel 2: m distribution
ax = axes[0, 1]
ax.hist(boot_params['m'], bins=30, color=COLORS['green'], alpha=0.6, density=True)
m_lo, m_hi = np.percentile(boot_params['m'], [2.5, 97.5])
ax.axvline(params['m'], color=COLORS['red'], linewidth=2, linestyle='--',
           label=f'Point est: {params["m"]:.3f}')
ax.axvline(m_lo, color=COLORS['amber'], linewidth=1.5, linestyle=':')
ax.axvline(m_hi, color=COLORS['amber'], linewidth=1.5, linestyle=':')
ax.set_title('Distribution of $m$ (Power-Law Exponent)', fontweight='bold')
ax.set_xlabel('$m$')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=1, frameon=False, fontsize=9)

# Panel 3: omega distribution
ax = axes[1, 0]
ax.hist(boot_params['omega'], bins=30, color=COLORS['purple'], alpha=0.6, density=True)
omega_lo, omega_hi = np.percentile(boot_params['omega'], [2.5, 97.5])
ax.axvline(params['omega'], color=COLORS['red'], linewidth=2, linestyle='--',
           label=f'Point est: {params["omega"]:.2f}')
ax.axvline(omega_lo, color=COLORS['amber'], linewidth=1.5, linestyle=':')
ax.axvline(omega_hi, color=COLORS['amber'], linewidth=1.5, linestyle=':')
ax.set_title('Distribution of $\\omega$ (Log-Frequency)', fontweight='bold')
ax.set_xlabel('$\\omega$')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=1, frameon=False, fontsize=9)

# Panel 4: m vs omega scatter
ax = axes[1, 1]
ax.scatter(boot_params['m'], boot_params['omega'], s=15, alpha=0.5,
           color=COLORS['blue'], label='Bootstrap samples')
ax.scatter([params['m']], [params['omega']], s=120, color=COLORS['red'],
           marker='*', zorder=5, label='Point estimate')
ax.set_title('$m$ vs $\\omega$ Joint Distribution', fontweight='bold')
ax.set_xlabel('$m$')
ax.set_ylabel('$\\omega$')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=2, frameon=False, fontsize=9)

for ax in axes.flatten():
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_facecolor('none')

fig.suptitle('Bootstrap Distributions of LPPL Parameters (n=200)', fontweight='bold', y=1.01)
fig.patch.set_alpha(0)
plt.tight_layout()
plt.show()

# Convert tc CI to calendar dates
tc_lo_date = btc.index[-1] + pd.offsets.BDay(max(0, int(np.round(tc_lo)) - N + 1))
tc_hi_date = btc.index[-1] + pd.offsets.BDay(max(0, int(np.round(tc_hi)) - N + 1))
tc_pt_date = btc.index[-1] + pd.offsets.BDay(max(0, int(np.round(params['tc'])) - N + 1))

print('95% Confidence Intervals:')
print(f'  tc:    [{tc_lo:.1f}, {tc_hi:.1f}] trading days')
print(f'  tc:    [{tc_lo_date.strftime("%Y-%m-%d")}, {tc_hi_date.strftime("%Y-%m-%d")}] calendar')
print(f'  m:     [{m_lo:.3f}, {m_hi:.3f}]')
print(f'  omega: [{omega_lo:.2f}, {omega_hi:.2f}]')

## Exercise 7: Multi-Window Stability

**Task:** Fit LPPL over multiple window lengths ending at the same date.  
If the estimated $t_c$ is stable across different window sizes, this increases confidence in the bubble signal.  
Windows that fail the 8 filter conditions are flagged.

In [ ]:
# Multi-window LPPL fitting
windows = [60, 90, 120, 150, 180, 200, 250, 300]
window_results = []

print('Multi-window LPPL fitting...')
for w in windows:
    if w > N:
        continue
    t_w = np.arange(w)
    y_w = log_price[-w:]
    try:
        p_w = fit_lppl(t_w, y_w, seed=42)
        conds = check_filter_conditions(p_w, w)
        n_pass = sum(c[1] for c in conds)
        all_pass = (n_pass == 8)
        # Convert tc to days-from-end-of-sample
        tc_from_end = p_w['tc'] - w
        window_results.append({
            'window': w, 'tc': p_w['tc'], 'tc_from_end': tc_from_end,
            'm': p_w['m'], 'omega': p_w['omega'], 'B': p_w['B'],
            'n_pass': n_pass, 'all_pass': all_pass
        })
        status = 'PASS (8/8)' if all_pass else f'PARTIAL ({n_pass}/8)'
        print(f'  Window {w:>4d}: tc={p_w["tc"]:>7.1f}  m={p_w["m"]:.3f}  '
              f'omega={p_w["omega"]:>5.2f}  {status}')
    except Exception as e:
        print(f'  Window {w:>4d}: FAILED ({e})')

wr_df = pd.DataFrame(window_results)

# Plot tc estimates vs window length
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: tc estimate vs window
colors_w = [COLORS['green'] if r['all_pass'] else COLORS['red'] for _, r in wr_df.iterrows()]
axes[0].scatter(wr_df['window'], wr_df['tc_from_end'], c=colors_w, s=100, zorder=5)
for _, r in wr_df.iterrows():
    axes[0].annotate(f'{r["n_pass"]}/8', (r['window'], r['tc_from_end']),
                     textcoords='offset points', xytext=(0, 12), ha='center', fontsize=9)
axes[0].axhline(0, color=COLORS['gray'], linestyle='--', linewidth=0.8)
axes[0].set_title('$t_c$ Estimate vs Window Length', fontweight='bold')
axes[0].set_xlabel('Window length (trading days)')
axes[0].set_ylabel('$t_c$ − end of sample (days ahead)')
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=COLORS['green'], label='All 8 filters pass'),
                   Patch(facecolor=COLORS['red'], label='Some filters fail')]
axes[0].legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, -0.15),
               ncol=2, frameon=False)
axes[0].spines[['top', 'right']].set_visible(False)
axes[0].set_facecolor('none')

# Panel 2: m and omega across windows
ax2 = axes[1]
ax2.plot(wr_df['window'], wr_df['m'], 'o-', color=COLORS['blue'], linewidth=1.5,
         markersize=8, label='$m$')
ax2_twin = ax2.twinx()
ax2_twin.plot(wr_df['window'], wr_df['omega'], 's-', color=COLORS['purple'], linewidth=1.5,
              markersize=8, label='$\\omega$')
ax2.set_title('Parameter Stability Across Windows', fontweight='bold')
ax2.set_xlabel('Window length (trading days)')
ax2.set_ylabel('$m$', color=COLORS['blue'])
ax2_twin.set_ylabel('$\\omega$', color=COLORS['purple'])
ax2.spines[['top']].set_visible(False)
ax2_twin.spines[['top']].set_visible(False)
ax2.set_facecolor('none')
lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax2_twin.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2,
           loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2, frameon=False)

fig.patch.set_alpha(0)
plt.tight_layout()
plt.show()

n_valid = wr_df['all_pass'].sum()
ci_score = n_valid / len(wr_df)
print(f'\nStability summary:')
print(f'  Windows tested: {len(wr_df)}')
print(f'  Windows passing all 8 filters: {n_valid}')
print(f'  Simple Confidence Index: {ci_score:.2f}')

## Exercise 8: LPPLS Confidence Indicator

**Task:** Implement a rolling LPPLS Confidence Indicator (CI).  
At each date $t$, fit LPPL over multiple windows ending at $t$ and count the fraction of fits that pass all 8 filter conditions.  
Plot the price with the CI overlay using traffic-light coloring:  
- **Green** (CI < 0.3): no significant bubble signal  
- **Amber** (0.3 ≤ CI < 0.6): emerging bubble signal  
- **Red** (CI ≥ 0.6): strong bubble warning

In [ ]:
# Rolling LPPLS Confidence Indicator
ci_windows = [60, 90, 120, 150, 180]
min_window = max(ci_windows)
step = 5  # compute CI every 5 trading days for speed

ci_dates = []
ci_values = []

eval_indices = list(range(min_window, N, step))
print(f'Computing LPPLS CI at {len(eval_indices)} dates (step={step})...')

for idx_count, end_idx in enumerate(eval_indices):
    n_valid_ci = 0
    n_total_ci = 0
    for w in ci_windows:
        if end_idx < w:
            continue
        t_w = np.arange(w)
        y_w = log_price[end_idx - w:end_idx]
        n_total_ci += 1
        try:
            p_ci = fit_lppl(t_w, y_w, seed=42)
            conds = check_filter_conditions(p_ci, w)
            if sum(c[1] for c in conds) == 8:
                n_valid_ci += 1
        except Exception:
            pass
    ci_val = n_valid_ci / max(n_total_ci, 1)
    ci_dates.append(btc.index[end_idx - 1])
    ci_values.append(ci_val)
    if (idx_count + 1) % 20 == 0:
        print(f'  {idx_count+1}/{len(eval_indices)} done')

ci_series = pd.Series(ci_values, index=ci_dates)
print(f'Done. CI computed at {len(ci_series)} dates.')

In [ ]:
# Plot price with CI overlay
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [2, 1]})

# Top panel: Price
ax1.plot(btc.index, btc['Price'], color=COLORS['blue'], linewidth=1)
ax1.set_title('BTC-USD Price with LPPLS Confidence Indicator', fontweight='bold')
ax1.set_ylabel('Price (USD)')
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
ax1.spines[['top', 'right']].set_visible(False)
ax1.set_facecolor('none')

# Bottom panel: CI with traffic-light coloring
for i in range(len(ci_series) - 1):
    ci_val = ci_series.iloc[i]
    if ci_val >= 0.6:
        color = COLORS['red']
    elif ci_val >= 0.3:
        color = COLORS['amber']
    else:
        color = COLORS['green']
    ax2.bar(ci_series.index[i], ci_val, width=step + 2,
            color=color, alpha=0.7)

ax2.axhline(0.3, color=COLORS['amber'], linestyle='--', linewidth=1, alpha=0.8)
ax2.axhline(0.6, color=COLORS['red'], linestyle='--', linewidth=1, alpha=0.8)
ax2.set_ylim(0, 1)
ax2.set_title('LPPLS Confidence Indicator', fontweight='bold')
ax2.set_ylabel('CI')
ax2.set_xlabel('Date')
ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))

# Legend
legend_elements = [
    Patch(facecolor=COLORS['green'], alpha=0.7, label='CI < 0.3 (No signal)'),
    Patch(facecolor=COLORS['amber'], alpha=0.7, label='0.3 <= CI < 0.6 (Emerging)'),
    Patch(facecolor=COLORS['red'], alpha=0.7, label='CI >= 0.6 (Strong warning)')
]
ax2.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, -0.25),
           ncol=3, frameon=False)
ax2.spines[['top', 'right']].set_visible(False)
ax2.set_facecolor('none')

fig.patch.set_alpha(0)
plt.tight_layout()
plt.show()

print(f'Maximum CI: {ci_series.max():.2f} on {ci_series.idxmax().strftime("%Y-%m-%d")}')
print(f'Dates with CI >= 0.6: {(ci_series >= 0.6).sum()}')
print(f'Dates with CI >= 0.3: {(ci_series >= 0.3).sum()}')

### Question 8.1

How does the LPPLS CI evolve over the Bitcoin bubble period?  
Does the CI rise before the peak price, providing an early warning signal?

**YOUR ANSWER HERE**

---
# Part III: PhD Extension

**Objectives:** Apply LPPL as a negative control on a non-bubble crash (COVID-19 exogenous shock) and design a risk management strategy based on the LPPLS Confidence Indicator.

## Exercise 9: Negative Control — COVID 2020

**Task:** Download S&P 500 data from Jan 2019 to Jun 2020.  
Attempt to fit LPPL to the pre-crash period.  
Show that the filter conditions FAIL — there is no valid endogenous bubble signal.  
This demonstrates that LPPL correctly does NOT produce a false alarm when the crash is caused by an exogenous shock (the COVID-19 pandemic) rather than endogenous bubble dynamics.

In [ ]:
# Download S&P 500 pre-COVID data
sp500 = yf.download('^GSPC', start='2019-01-01', end='2020-06-30', progress=False)
sp500 = sp500[['Close']].dropna()
sp500.columns = ['Price']

# Use data up to Feb 2020 (before crash)
sp500_pre = sp500.loc[:'2020-02-19']  # S&P 500 peaked on Feb 19, 2020
sp500_pre['LogPrice'] = np.log(sp500_pre['Price'])

log_price_sp = sp500_pre['LogPrice'].values
N_sp = len(log_price_sp)
t_sp = np.arange(N_sp)

print(f'S&P 500 pre-COVID sample: {sp500_pre.index[0].strftime("%Y-%m-%d")} to '
      f'{sp500_pre.index[-1].strftime("%Y-%m-%d")} ({N_sp} observations)')

# Fit LPPL
print('\nFitting LPPL to S&P 500 pre-COVID data...')
params_sp = fit_lppl(t_sp, log_price_sp, seed=42)

# Generate fitted values
y_fit_sp = lppl_func(t_sp, params_sp['tc'], params_sp['m'], params_sp['omega'],
                      params_sp['A'], params_sp['B'], params_sp['C1'], params_sp['C2'])

# Check filter conditions
conditions_sp = check_filter_conditions(params_sp, N_sp)

print('\nLPPL Fitted Parameters (S&P 500 pre-COVID):')
print('=' * 45)
for key in ['A', 'B', 'C', 'm', 'omega', 'phi', 'tc']:
    print(f'  {key:>5s} = {params_sp[key]:>10.4f}')

print('\nFilter Conditions:')
print('=' * 75)
n_pass_sp = 0
for name, passed, value, bound in conditions_sp:
    status = 'PASS' if passed else 'FAIL'
    n_pass_sp += int(passed)
    print(f'  [{"+ " if passed else "X "}] {status}  {name}')
    print(f'         Value: {value}  |  Required: {bound}')
print('=' * 75)
print(f'Result: {n_pass_sp}/8 conditions passed')

In [ ]:
# Plot comparison: BTC bubble (valid) vs S&P COVID (invalid)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: BTC bubble fit
axes[0].plot(btc.index, log_price, color=COLORS['blue'], linewidth=1, alpha=0.7,
             label='Observed')
axes[0].plot(btc.index, y_fit, color=COLORS['red'], linewidth=2, label='LPPL fit')
n_pass_btc = sum(c[1] for c in check_filter_conditions(params, N))
axes[0].set_title(f'BTC 2021 Bubble \u2014 {n_pass_btc}/8 Filters Pass', fontweight='bold')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Log-price')
axes[0].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2, frameon=False)

# Panel 2: S&P COVID fit
axes[1].plot(sp500_pre.index, log_price_sp, color=COLORS['blue'], linewidth=1, alpha=0.7,
             label='Observed')
axes[1].plot(sp500_pre.index, y_fit_sp, color=COLORS['red'], linewidth=2, label='LPPL fit')
axes[1].set_title(f'S&P 500 pre-COVID \u2014 {n_pass_sp}/8 Filters Pass', fontweight='bold')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Log-price')
axes[1].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
axes[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2, frameon=False)

for ax in axes:
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_facecolor('none')

fig.suptitle('LPPL Bubble Detection: True Bubble vs Exogenous Crash', fontweight='bold', y=1.01)
fig.patch.set_alpha(0)
plt.tight_layout()
plt.show()

print('\nConclusion:')
print('  - BTC 2021: LPPL fits well and filter conditions are satisfied -> valid bubble signal')
print('  - S&P pre-COVID: LPPL does NOT produce a valid signal -> correct negative')
print('  - The COVID crash was an exogenous shock, not an endogenous bubble collapse')
print('  - This validates LPPL as a tool that distinguishes endogenous from exogenous crashes')

### Question 9.1

Why is negative control testing essential for any bubble detection methodology?  
Can you think of other historical crashes that were exogenous (not driven by bubble dynamics)?

**YOUR ANSWER HERE**

## Exercise 10: Risk Management Application

**Task:** Given the CI time series from Exercise 8, implement a dynamic position-sizing strategy.  
When the CI is high (bubble warning), reduce exposure. When the CI is low, maintain full exposure.  

Position-sizing rule: $\text{position} = \max(0.1,\; 1 - 1.5 \cdot CI^{1.5})$  

Compare buy-and-hold vs the LPPL-adjusted strategy in terms of cumulative returns, drawdowns, Sharpe ratio, and maximum drawdown.

In [ ]:
# Build daily CI series by forward-filling the step-based CI
ci_daily = ci_series.reindex(btc.index).ffill().fillna(0)

# Position sizing: position = max(0.1, 1 - 1.5 * CI^1.5)
position = np.maximum(0.1, 1 - 1.5 * ci_daily.values**1.5)

# Compute returns
daily_returns = btc['Price'].pct_change().fillna(0).values

# Buy and hold
bh_cum = np.cumprod(1 + daily_returns)

# LPPL-adjusted
lppl_returns = position * daily_returns
lppl_cum = np.cumprod(1 + lppl_returns)

# Drawdowns
def compute_drawdown(cum_returns):
    peak = np.maximum.accumulate(cum_returns)
    dd = (cum_returns - peak) / peak
    return dd

dd_bh = compute_drawdown(bh_cum)
dd_lppl = compute_drawdown(lppl_cum)

fig, axes = plt.subplots(3, 1, figsize=(14, 10), gridspec_kw={'height_ratios': [2, 1, 1]})

# Panel 1: Cumulative returns
axes[0].plot(btc.index, bh_cum, color=COLORS['blue'], linewidth=1.5,
             label='Buy & Hold')
axes[0].plot(btc.index, lppl_cum, color=COLORS['red'], linewidth=1.5,
             label='LPPL-Adjusted')
axes[0].set_title('Cumulative Returns: Buy & Hold vs LPPL-Adjusted', fontweight='bold')
axes[0].set_ylabel('Cumulative return (x)')
axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=2, frameon=False)
axes[0].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
axes[0].spines[['top', 'right']].set_visible(False)
axes[0].set_facecolor('none')

# Panel 2: Position size
axes[1].fill_between(btc.index, position, alpha=0.4, color=COLORS['green'],
                     label='Position size')
axes[1].plot(btc.index, position, color=COLORS['green'], linewidth=1)
axes[1].set_title('Dynamic Position Size', fontweight='bold')
axes[1].set_ylabel('Position')
axes[1].set_ylim(0, 1.1)
axes[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=1, frameon=False)
axes[1].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
axes[1].spines[['top', 'right']].set_visible(False)
axes[1].set_facecolor('none')

# Panel 3: Drawdowns
axes[2].fill_between(btc.index, dd_bh, 0, alpha=0.4, color=COLORS['blue'],
                     label='Buy & Hold')
axes[2].fill_between(btc.index, dd_lppl, 0, alpha=0.4, color=COLORS['red'],
                     label='LPPL-Adjusted')
axes[2].set_title('Drawdowns', fontweight='bold')
axes[2].set_ylabel('Drawdown')
axes[2].set_xlabel('Date')
axes[2].legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=2, frameon=False)
axes[2].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
axes[2].xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
axes[2].spines[['top', 'right']].set_visible(False)
axes[2].set_facecolor('none')

fig.patch.set_alpha(0)
plt.tight_layout()
plt.show()

In [ ]:
# Performance metrics
def sharpe_ratio(returns, rf=0):
    excess = returns - rf / 252
    return np.sqrt(252) * excess.mean() / excess.std() if excess.std() > 0 else 0

sharpe_bh = sharpe_ratio(daily_returns)
sharpe_lppl = sharpe_ratio(lppl_returns)
max_dd_bh = dd_bh.min()
max_dd_lppl = dd_lppl.min()
total_ret_bh = bh_cum[-1] - 1
total_ret_lppl = lppl_cum[-1] - 1

print('Performance Comparison')
print('=' * 55)
print(f'{"Metric":>25s} {"Buy & Hold":>14s} {"LPPL-Adjusted":>14s}')
print('-' * 55)
print(f'{"Total Return":>25s} {total_ret_bh*100:>13.1f}% {total_ret_lppl*100:>13.1f}%')
print(f'{"Annualized Sharpe":>25s} {sharpe_bh:>14.3f} {sharpe_lppl:>14.3f}')
print(f'{"Max Drawdown":>25s} {max_dd_bh*100:>13.1f}% {max_dd_lppl*100:>13.1f}%')
print(f'{"Daily Volatility":>25s} {np.std(daily_returns)*100:>13.2f}% {np.std(lppl_returns)*100:>13.2f}%')
print(f'{"Mean Position Size":>25s} {"100.0%":>14s} {np.mean(position)*100:>13.1f}%')

print('\nInterpretation:')
print('  The LPPL-adjusted strategy reduces position size when bubble risk is elevated.')
print('  This typically reduces drawdowns and volatility at the cost of some upside.')
print('  The Sharpe ratio comparison shows the risk-adjusted benefit of LPPL monitoring.')

### Question 10.1

What are the limitations of using LPPL for real-time risk management?  
How would transaction costs and the look-ahead bias in parameter estimation affect the strategy?

**YOUR ANSWER HERE**

---
## Summary: Chapter 13 Seminar

| Part | Exercise | Topic | Key Concepts |
|------|----------|-------|-------------|
| I — Year 3 | 1 | Super-exponential growth | Linear vs quadratic vs exponential fit, $R^2$ comparison |
| I — Year 3 | 2 | LPPL equation | Power-law trend, log-periodic oscillations, critical time $t_c$ |
| I — Year 3 | 3 | LPPL fitting | Partial linearization, differential evolution, OLS for linear params |
| II — Master | 4 | Filter conditions | 8 conditions for valid bubble signal, parameter bounds |
| II — Master | 5 | Benchmark models | AIC/BIC comparison, model selection, parsimony vs fit |
| II — Master | 6 | Bootstrap CI | Residual resampling, 95% CI for $t_c$, joint parameter uncertainty |
| II — Master | 7 | Multi-window stability | Robustness across fitting windows, consistency of $t_c$ |
| II — Master | 8 | LPPLS Confidence Indicator | Rolling CI, traffic-light classification, early warning |
| III — PhD | 9 | Negative control | COVID crash, exogenous vs endogenous, false alarm rate |
| III — PhD | 10 | Risk management | Dynamic position sizing, drawdown reduction, Sharpe ratio |

### Key Takeaways

- **Super-exponential growth** is the hallmark of a bubble: log-price grows faster than linearly, driven by positive feedback loops and herding
- **LPPL model** decomposes bubble dynamics into a power-law trend and log-periodic oscillations that accelerate toward a critical time $t_c$
- **Partial linearization** reduces the 7-parameter estimation to a 3D nonlinear search (over $t_c$, $m$, $\omega$) plus 4D OLS
- **8 filter conditions** ensure that fitted parameters are physically meaningful and consistent with bubble theory
- **Bootstrap** and **multi-window** analyses quantify uncertainty and assess robustness of the bubble signal
- **LPPLS Confidence Indicator** provides a real-time, traffic-light summary of bubble risk at each date
- **Negative controls** (e.g., COVID crash) validate that LPPL does not produce false alarms for exogenous shocks
- **Risk management**: the CI can drive dynamic position sizing to reduce exposure during elevated bubble risk